# LOO summary table (DEG 200)

Builds a NeurIPS-ready summary table from `results/loo_summary_deg_200.csv`.

- Aggregates across seeds and held-out cell types (mean ± std).
- Bolds the best performer per metric (direction-aware).
- Renames `baseline` to `mean shift`.
- Annotates columns with ↑ (higher is better) or ↓ (lower is better).

In [1]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from pathlib import Path

In [3]:
DATASET_NAME = "merfish"
N_DEG = 50
CSV_PATH = f"../results/loo_summary_{DATASET_NAME}_DEG_{N_DEG}.csv"
OUT_DIR = Path("../results")
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(CSV_PATH, engine="python")
df.head()

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,direction_match_gt,mixing_index,edistance_global,edistance_local,edistance_pca,edistance_pca_log,rmse,mse_lfc
0,C57BL6J-2.036,baseline,GABAergic neuron_Fiber_tracts,50,0.574783,0.676753,0.00,0.0,0.00,0.08,0.019022,42.347551,43.229744,1522.121790,27.050253,613.618408,398.293091
1,C57BL6J-2.036,baseline,GABAergic neuron_Isocortex,50,0.656297,0.672887,0.20,0.9,0.18,0.60,0.293658,48.110075,52.501921,1118.618372,29.387819,1490.194336,180.199417
2,C57BL6J-2.036,baseline,astrocyte_Fiber_tracts,50,0.301952,0.354013,0.18,1.0,0.18,0.82,0.069700,38.657775,37.840701,1407.842004,17.100988,1868.877075,152.413559
3,C57BL6J-2.036,baseline,astrocyte_Isocortex,50,0.774857,0.715261,0.26,1.0,0.26,0.84,0.000000,64.602134,66.657219,2111.273730,36.885138,2971.760254,154.598236
4,C57BL6J-2.036,baseline,endothelial cell_Fiber_tracts,50,0.742315,0.662692,0.06,1.0,0.06,0.86,0.003236,43.568983,43.794907,1866.005750,21.782019,1668.539429,96.198738


In [4]:
# Rename holdout_celltype by replacing the last '_' in with '-'
if DATASET_NAME == "merfish":
    df["holdout_celltype"] = df["holdout_celltype"].str.replace("Fiber_tracts", "Fiber-tracts", regex=False)
else:
    df["holdout_celltype"] = df["holdout_celltype"].str.replace("T_cell", "T-cell", regex=False)

In [5]:
# Split holdout_celltype on '_' and place everything in [0] as holdout_celltype, and everything in [1:] as perturbation
df["perturbation"] = df["holdout_celltype"].apply(lambda x: "".join(x.split("_")[-1]) if len(x.split("_")) > 1 else "")
df["holdout_celltype"] = df["holdout_celltype"].apply(lambda x: "-".join(x.split("_")[:-1]))
df

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,direction_match_gt,mixing_index,edistance_global,edistance_local,edistance_pca,edistance_pca_log,rmse,mse_lfc,perturbation
0,C57BL6J-2.036,baseline,GABAergic neuron,50,0.574783,0.676753,0.00,0.0,0.00,0.08,0.019022,42.347551,43.229744,1522.121790,27.050253,613.618408,398.293091,Fiber-tracts
1,C57BL6J-2.036,baseline,GABAergic neuron,50,0.656297,0.672887,0.20,0.9,0.18,0.60,0.293658,48.110075,52.501921,1118.618372,29.387819,1490.194336,180.199417,Isocortex
2,C57BL6J-2.036,baseline,astrocyte,50,0.301952,0.354013,0.18,1.0,0.18,0.82,0.069700,38.657775,37.840701,1407.842004,17.100988,1868.877075,152.413559,Fiber-tracts
3,C57BL6J-2.036,baseline,astrocyte,50,0.774857,0.715261,0.26,1.0,0.26,0.84,0.000000,64.602134,66.657219,2111.273730,36.885138,2971.760254,154.598236,Isocortex
4,C57BL6J-2.036,baseline,endothelial cell,50,0.742315,0.662692,0.06,1.0,0.06,0.86,0.003236,43.568983,43.794907,1866.005750,21.782019,1668.539429,96.198738,Fiber-tracts
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
293,C57BL6J-2.041,spatialprop-pert,astrocyte,50,0.716591,0.743897,0.08,1.0,0.08,0.90,0.064824,46.138379,53.464196,412.151459,17.192637,2914.762200,35.410297,Fiber-tracts
294,C57BL6J-2.041,spatialprop-pert,oligodendrocyte,50,0.399760,0.634266,0.12,1.0,0.12,0.96,0.156538,57.556989,66.563298,504.260925,18.777699,1367.118900,130.861150,Isocortex
295,C57BL6J-2.041,spatialprop-pert,oligodendrocyte,50,0.301609,0.667436,0.06,1.0,0.06,0.80,0.078554,51.789951,59.460059,546.684589,19.268379,6746.678000,20.537620,Fiber-tracts
296,C57BL6J-2.041,spatialprop-pert,endothelial cell,50,0.801873,0.870983,0.32,1.0,0.32,0.98,0.008873,56.686428,62.951312,474.125043,19.853804,2641.575400,64.354140,Isocortex


In [6]:
# Apply log10 on rmse
import numpy as np
df['rmse'] = np.log10(df['rmse'])

In [7]:
# Apply square root to mse_lfc
df['rmse_lfc'] = np.sqrt(df['mse_lfc'])

In [40]:
#df = df[df["model_name"] != "cellina-gat-pert"]

In [8]:
# Metrics we want in the table and whether higher (+1) or lower (-1) is better.
METRICS = {
    "spearman": +1,
    "pearson": +1,
    #"precision": +1,
    "direction_match_k": +1,
    #"edistance_local": -1,
    #"edistance_global": -1,
    "rmse": -1,
    "rmse_lfc": -1,
    #"avg_rank": -1,
}

PRETTY_METRIC = {
    "spearman": r"Spearman $\uparrow$",
    "pearson": r"Pearson $\uparrow$",
    #"precision": r"Precision $\uparrow$",
    "direction_match_k": r"Signed Precision $\uparrow$",
    #"edistance_local": r"E-dist (local) $\downarrow$",
    #"edistance_global": r"E-dist (global) $\downarrow$",
    "rmse": r"RMSE\textsubscript{counts} $\downarrow$",
    "rmse_lfc": r"RMSE\textsubscript{LFC} $\downarrow$",
    #"avg_rank": r"Avg. Rank $\downarrow$",
}

# Model display order + renaming (baseline -> mean shift).
MODEL_RENAME = {
    "baseline": "mean shift",
    "cpa": "CPA",
    "scgen": "scGen",
    "spatialprop": "SpatialProp",
    "mintflow": "MintFlow",
    "cellina-ablated": "cellina (ablated)",
    "cellina-graph": "cellina-GAT",
    "cellina": "cellina",
    "cellina-pert": r"cellina$_{node-pert}$",
    #"cellina-gat-pert": r"cellina-GAT$_{node-pert}$",
    "spatialprop-pert": r"SpatialProp$_{node-pert}$"
}
MODEL_ORDER = [
    "mean shift",
    "CPA",
    "scGen",
    "SpatialProp",
    "MintFlow",
    "cellina (ablated)",
    "cellina-GAT",
    "cellina",
    r"cellina$_{node-pert}$",
    #r"cellina-GAT$_{node-pert}$",
    r"SpatialProp$_{node-pert}$",
]

df["model_name"] = df["model_name"].map(lambda x: MODEL_RENAME.get(x, x))
df["model_name"].unique()

array(['mean shift', 'cellina (ablated)', 'cellina', 'cellina-GAT', 'CPA',
       'scGen', 'MintFlow', 'SpatialProp', 'cellina$_{node-pert}$',
       'SpatialProp$_{node-pert}$'], dtype=object)

In [20]:
# Two-step aggregation:
#   1) within each slide, average over held-out cell types
#   2) across slides, compute mean ± std
agg = (
    df.groupby(["perturbation", "model_name"])[list(METRICS)]
    .agg(["mean", "std"])
)

# Ensure a consistent model order within each perturbation. Build a
# MultiIndex with (perturbation, model) in the desired display order and
# reindex the aggregated table to that index.
perturbations = list(df["perturbation"].dropna().unique())
# keep stable ordering; treat empty string as its own group if present
if any(p == "" for p in perturbations):
    perturbations = sorted(perturbations, key=lambda x: (x != "", x))
else:
    perturbations = sorted(perturbations)

desired_index = pd.MultiIndex.from_product(
    [perturbations, MODEL_ORDER], names=["perturbation", "model_name"]
)
agg = agg.reindex(desired_index)
agg

spearman             pearson  \
                                            mean       std      mean   
perturbation model_name                                                
Fiber-tracts mean shift                 0.351411  0.271832  0.360952   
             CPA                        0.584916  0.206990  0.798208   
             scGen                      0.496118  0.229757  0.715875   
             SpatialProp                0.495657  0.188576  0.666168   
             MintFlow                   0.546366  0.229092  0.779929   
             cellina (ablated)          0.584583  0.215306  0.797372   
             cellina-GAT                0.644486  0.153088  0.805590   
             cellina                    0.659409  0.213460  0.826450   
             cellina$_{node-pert}$      0.578808  0.225657  0.803836   
             SpatialProp$_{node-pert}$  0.492821  0.184935  0.664690   
Isocortex    mean shift                 0.594199  0.188672  0.575376   
             CPA                        0.781948  0.170337  0.837973   
             scGen                      0.719862  0.182515  0.820247   
             SpatialProp                0.704675  0.122950  0.744314   
             MintFlow                   0.772120  0.183311  0.836694   
             cellina (ablated)          0.789042  0.162555  0.841151   
             cellina-GAT                0.852629  0.110558  0.888902   
             cellina                    0.808256  0.177170  0.858398   
             cellina$_{node-pert}$      0.736579  0.194731  0.787128   
             SpatialProp$_{node-pert}$  0.705489  0.123023  0.744781   

                                                 direction_match_k            \
                                             std              mean       std   
perturbation model_name                                                        
Fiber-tracts mean shift                 0.238833          0.068000  0.069200   
             CPA                        0.149858          0.309333  0.125554   
             scGen                      0.201174          0.149333  0.090354   
             SpatialProp                0.177190          0.081333  0.069884   
             MintFlow                   0.179957          0.205333  0.162035   
             cellina (ablated)          0.141151          0.280000  0.141825   
             cellina-GAT                0.157889          0.394286  0.181350   
             cellina                    0.154841          0.453333  0.212289   
             cellina$_{node-pert}$      0.151911          0.278667  0.177517   
             SpatialProp$_{node-pert}$  0.176668          0.081333  0.069884   
Isocortex    mean shift                 0.191781          0.250667  0.081369   
             CPA                        0.172484          0.534667  0.136584   
             scGen                      0.142527          0.268000  0.161917   
             SpatialProp                0.092886          0.084000  0.079714   
             MintFlow                   0.164855          0.366667  0.161186   
             cellina (ablated)          0.163441          0.501333  0.139277   
             cellina-GAT                0.143880          0.657143  0.098873   
             cellina                    0.170373          0.588000  0.138935   
             cellina$_{node-pert}$      0.185955          0.353333  0.171325   
             SpatialProp$_{node-pert}$  0.092850          0.084000  0.079714   

                                            rmse             rmse_lfc  \
                                            mean       std       mean   
perturbation model_name                                                 
Fiber-tracts mean shift                 3.331005  0.295626  12.960565   
             CPA                        3.636333  0.356673   7.645226   
             scGen                      3.534082  0.355144   7.510745   
             SpatialProp                3.272859  0.397282   7.181114   
             MintFlow          

In [21]:
def find_best(agg: pd.DataFrame) -> dict:
    best = {}
    if isinstance(agg.index, pd.MultiIndex):
        # agg index: (perturbation, model_name)
        # collect perturbations preserving order
        raw = [p for p, _ in agg.index.tolist()]
        seen = set()
        perturbations = [p for p in raw if not (p in seen or seen.add(p))]
        for metric, direction in METRICS.items():
            for pert in perturbations:
                try:
                    means = agg.loc[pert][(metric, "mean")].dropna()
                except KeyError:
                    continue
                if means.empty:
                    continue
                best[(pert, metric)] = means.idxmax() if direction > 0 else means.idxmin()
    else:
        for metric, direction in METRICS.items():
            means = agg[(metric, "mean")].dropna()
            if means.empty:
                continue
            best[metric] = means.idxmax() if direction > 0 else means.idxmin()
    return best


def format_cell(mean, std, bold, na_str="--"):
    if pd.isna(mean):
        return na_str
    s = f"{mean:.2f} $\\pm$ {std:.2f}" if not pd.isna(std) else f"{mean:.2f}"
    return f"\\textbf{{{s}}}" if bold else s

In [22]:
best = find_best(agg)
table = pd.DataFrame(index=agg.index, columns=[PRETTY_METRIC[m] for m in METRICS])

for metric in METRICS:
    col = PRETTY_METRIC[metric]
    for model in agg.index:
        pert, model_name = model

        mean = agg.loc[model, (metric, "mean")]
        std = agg.loc[model, (metric, "std")]

        bold = (best.get((pert, metric)) == model_name)

        table.loc[model, col] = format_cell(mean, std, bold)
table

Spearman $\uparrow$  \
perturbation model_name                                            
Fiber-tracts mean shift                          0.35 $\pm$ 0.27   
             CPA                                 0.58 $\pm$ 0.21   
             scGen                               0.50 $\pm$ 0.23   
             SpatialProp                         0.50 $\pm$ 0.19   
             MintFlow                            0.55 $\pm$ 0.23   
             cellina (ablated)                   0.58 $\pm$ 0.22   
             cellina-GAT                         0.64 $\pm$ 0.15   
             cellina                    \textbf{0.66 $\pm$ 0.21}   
             cellina$_{node-pert}$               0.58 $\pm$ 0.23   
             SpatialProp$_{node-pert}$           0.49 $\pm$ 0.18   
Isocortex    mean shift                          0.59 $\pm$ 0.19   
             CPA                                 0.78 $\pm$ 0.17   
             scGen                               0.72 $\pm$ 0.18   
             SpatialProp                         0.70 $\pm$ 0.12   
             MintFlow                            0.77 $\pm$ 0.18   
             cellina (ablated)                   0.79 $\pm$ 0.16   
             cellina-GAT                \textbf{0.85 $\pm$ 0.11}   
             cellina                             0.81 $\pm$ 0.18   
             cellina$_{node-pert}$               0.74 $\pm$ 0.19   
             SpatialProp$_{node-pert}$           0.71 $\pm$ 0.12   

                                              Pearson $\uparrow$  \
perturbation model_name                                            
Fiber-tracts mean shift                          0.36 $\pm$ 0.24   
             CPA                                 0.80 $\pm$ 0.15   
             scGen                               0.72 $\pm$ 0.20   
             SpatialProp                         0.67 $\pm$ 0.18   
             MintFlow                            0.78 $\pm$ 0.18   
             cellina (ablated)                   0.80 $\pm$ 0.14   
             cellina-GAT                         0.81 $\pm$ 0.16   
             cellina                    \textbf{0.83 $\pm$ 0.15}   
             cellina$_{node-pert}$               0.80 $\pm$ 0.15   
             SpatialProp$_{node-pert}$           0.66 $\pm$ 0.18   
Isocortex    mean shift                          0.58 $\pm$ 0.19   
             CPA                                 0.84 $\pm$ 0.17   
             scGen                               0.82 $\pm$ 0.14   
             SpatialProp                         0.74 $\pm$ 0.09   
             MintFlow                            0.84 $\pm$ 0.16   
             cellina (ablated)                   0.84 $\pm$ 0.16   
             cellina-GAT                \textbf{0.89 $\pm$ 0.14}   
             cellina                             0.86 $\pm$ 0.17   
             cellina$_{node-pert}$               0.79 $\pm$ 0.19   
             SpatialProp$_{node-pert}$           0.74 $\pm$ 0.09   

                                       Signed Precision $\uparrow$  \
perturbation model_name                                              
Fiber-tracts mean shift                            0.07 $\pm$ 0.07   
             CPA                                   0.31 $\pm$ 0.13   
             scGen                                 0.15 $\pm$ 0.09   
             SpatialProp                           0.08 $\pm$ 0.07   
             MintFlow                              0.21 $\pm$ 0.16   
             cellina (ablated)                     0.28 $\pm$ 0.14   
             cellina-GAT                           0.39 $\pm$ 0.18   
             cellina                      \textbf{0.45 $\pm$ 0.21}   
             cellina$_{node-pert}$                 0.28 $\pm$ 0.18   
             SpatialProp$_{node-pert}$             0.08 $\pm$ 0.07   
Isocortex    mean shift                            0.25 $\pm$ 0.08   
             CPA                                   0.53 $\pm$ 0.14   
             scGen                                 0.27 $\pm$ 0.16   
 

In [23]:
def insert_midrule_perts(latex, table):
    groups = table.index.get_level_values(0)
    boundaries = [i for i in range(1, len(groups)) if groups[i] != groups[i - 1]]

    lines = latex.splitlines()
    new_lines = []

    row_idx = 0

    for line in lines:
        new_lines.append(line)

        # detect data rows (skip header)
        if "&" in line and "\\\\" in line:
            if row_idx in boundaries:
                new_lines.append(r"\midrule")
            row_idx += 1

    latex = "\n".join(new_lines)
    return latex

In [24]:
is_multi_pert = table.index.get_level_values(0).nunique() > 1

if is_multi_pert:
    latex = table.to_latex(
        index_names=False, # removes the extra header row
        escape=False,
        header=False,
        column_format="ll" + "c" * table.shape[1],  # two index levels → 'll'
        multirow=True,  # for multiple perturbations
        caption=(
            f"Leave-one-celltype-out performance (top {N_DEG} DEGs). For each slide we "
            "first average over the held-out cell types, then report mean $\\pm$ std "
            f"across {df['sid'].nunique()} slides. Best per metric in \\textbf{{bold}}."
        ),
        label=f"tab:loo_summary_{DATASET_NAME}",
    )

    latex = latex.replace(r"\cline{1-7}", "")
    latex = latex.replace("\\begin{table}", "\\begin{table}[t]\n\\centering")
    latex = latex.replace(r"\multirow[t]", r"\multirow[c]")
    header = "Holdout \\\ perturbation & Method & " + " & ".join(table.columns) + r" \\"
    latex = latex.replace(
    "\\begin{tabular}{ll" + "c" * table.shape[1] + "}",
    "\\begin{tabular}{ll" + "c" * table.shape[1] + "}\n\\toprule\n" + header
    )
    latex = latex.replace(r"\midrule", "")
    latex = insert_midrule_perts(latex, table)
else:
    flat = table.reset_index(level=0, drop=True)

    latex = flat.to_latex(
        index=True,
        index_names=False,
        escape=False,
        column_format="l" + "c" * flat.shape[1],
        caption=(
            f"Leave-one-celltype-out performance (top {N_DEG} DEGs). "
            "Mean $\\pm$ std across slides. Best per metric in \\textbf{bold}."
        ),
        label=f"tab:loo_summary_{DATASET_NAME}",
    )
    latex = latex.replace("\\begin{table}", "\\begin{table}[t]\n\\centering")

print(latex)

(OUT_DIR / f"loo_summary_{DATASET_NAME}_DEG_{N_DEG}_table.tex").write_text(latex)

\begin{table}[t]
\centering
\caption{Leave-one-celltype-out performance (top 50 DEGs). For each slide we first average over the held-out cell types, then report mean $\pm$ std across 3 slides. Best per metric in \textbf{bold}.}
\label{tab:loo_summary_merfish}
\begin{tabular}{llccccc}
\toprule
Holdout \\ perturbation & Method & Spearman $\uparrow$ & Pearson $\uparrow$ & Signed Precision $\uparrow$ & RMSE\textsubscript{counts} $\downarrow$ & RMSE\textsubscript{LFC} $\downarrow$ \\
\toprule

\multirow[c]{10}{*}{Fiber-tracts} & mean shift & 0.35 $\pm$ 0.27 & 0.36 $\pm$ 0.24 & 0.07 $\pm$ 0.07 & 3.33 $\pm$ 0.30 & 12.96 $\pm$ 3.90 \\
 & CPA & 0.58 $\pm$ 0.21 & 0.80 $\pm$ 0.15 & 0.31 $\pm$ 0.13 & 3.64 $\pm$ 0.36 & 7.65 $\pm$ 5.81 \\
 & scGen & 0.50 $\pm$ 0.23 & 0.72 $\pm$ 0.20 & 0.15 $\pm$ 0.09 & 3.53 $\pm$ 0.36 & 7.51 $\pm$ 5.14 \\
 & SpatialProp & 0.50 $\pm$ 0.19 & 0.67 $\pm$ 0.18 & 0.08 $\pm$ 0.07 & 3.27 $\pm$ 0.40 & \textbf{7.18 $\pm$ 3.39} \\
 & MintFlow & 0.55 $\pm$ 0.23 & 0.78 $\pm$ 0.1

2873

In [ ]:
# Markdown preview (arrows rendered as unicode so they look right in the notebook).
md_metric = {
    "spearman": "Spearman ↑",
    "pearson": "Pearson ↑",
    "precision": "Precision ↑",
    "direction_match_k": "Direction Match (K) ↑",
    "edistance_local": "E-dist (local) ↓",
    #"rmse": "RMSE ↓",
    "avg_rank": "Avg. Rank ↓",
}

md_table = pd.DataFrame(index=agg.index, columns=[md_metric[m] for m in METRICS])
for metric in METRICS:
    col = md_metric[metric]
    for model in agg.index:
        mean = agg.loc[model, (metric, "mean")]
        std = agg.loc[model, (metric, "std")]
        if pd.isna(mean):
            md_table.loc[model, col] = "--"
            continue
        s = f"{mean:.3f} ± {std:.3f}"
        md_table.loc[model, col] = f"**{s}**" if best.get(metric) == model else s

md_table.index.name = "Method"
print(md_table.to_markdown())

***

In [14]:
agg

spearman             pearson  \
                                            mean       std      mean   
perturbation model_name                                                
Fiber-tracts mean shift                 0.351411  0.271832  0.360952   
             CPA                        0.584916  0.206990  0.798208   
             scGen                      0.496118  0.229757  0.715875   
             SpatialProp                0.495657  0.188576  0.666168   
             MintFlow                   0.546366  0.229092  0.779929   
             cellina (ablated)          0.584583  0.215306  0.797372   
             cellina-GAT                0.644486  0.153088  0.805590   
             cellina                    0.659409  0.213460  0.826450   
             cellina$_{node-pert}$      0.578808  0.225657  0.803836   
             SpatialProp$_{node-pert}$  0.492821  0.184935  0.664690   
Isocortex    mean shift                 0.594199  0.188672  0.575376   
             CPA                        0.781948  0.170337  0.837973   
             scGen                      0.719862  0.182515  0.820247   
             SpatialProp                0.704675  0.122950  0.744314   
             MintFlow                   0.772120  0.183311  0.836694   
             cellina (ablated)          0.789042  0.162555  0.841151   
             cellina-GAT                0.852629  0.110558  0.888902   
             cellina                    0.808256  0.177170  0.858398   
             cellina$_{node-pert}$      0.736579  0.194731  0.787128   
             SpatialProp$_{node-pert}$  0.705489  0.123023  0.744781   

                                                 direction_match_k            \
                                             std              mean       std   
perturbation model_name                                                        
Fiber-tracts mean shift                 0.238833          0.068000  0.069200   
             CPA                        0.149858          0.309333  0.125554   
             scGen                      0.201174          0.149333  0.090354   
             SpatialProp                0.177190          0.081333  0.069884   
             MintFlow                   0.179957          0.205333  0.162035   
             cellina (ablated)          0.141151          0.280000  0.141825   
             cellina-GAT                0.157889          0.394286  0.181350   
             cellina                    0.154841          0.453333  0.212289   
             cellina$_{node-pert}$      0.151911          0.278667  0.177517   
             SpatialProp$_{node-pert}$  0.176668          0.081333  0.069884   
Isocortex    mean shift                 0.191781          0.250667  0.081369   
             CPA                        0.172484          0.534667  0.136584   
             scGen                      0.142527          0.268000  0.161917   
             SpatialProp                0.092886          0.084000  0.079714   
             MintFlow                   0.164855          0.366667  0.161186   
             cellina (ablated)          0.163441          0.501333  0.139277   
             cellina-GAT                0.143880          0.657143  0.098873   
             cellina                    0.170373          0.588000  0.138935   
             cellina$_{node-pert}$      0.185955          0.353333  0.171325   
             SpatialProp$_{node-pert}$  0.092850          0.084000  0.079714   

                                            rmse             rmse_lfc  \
                                            mean       std       mean   
perturbation model_name                                                 
Fiber-tracts mean shift                 3.331005  0.295626  12.960565   
             CPA                        3.636333  0.356673   7.645226   
             scGen                      3.534082  0.355144   7.510745   
             SpatialProp                3.272859  0.397282   7.181114   
             MintFlow          

In [15]:
df_avg = agg.groupby(level="model_name").mean()

In [16]:
df_avg = df_avg.loc[MODEL_ORDER]
df_avg

spearman             pearson            \
                               mean       std      mean       std   
model_name                                                          
mean shift                 0.472805  0.230252  0.468164  0.215307   
CPA                        0.683432  0.188663  0.818090  0.161171   
scGen                      0.607990  0.206136  0.768061  0.171850   
SpatialProp                0.600166  0.155763  0.705241  0.135038   
MintFlow                   0.659243  0.206201  0.808311  0.172406   
cellina (ablated)          0.686812  0.188930  0.819261  0.152296   
cellina-GAT                0.748558  0.131823  0.847246  0.150885   
cellina                    0.733833  0.195315  0.842424  0.162607   
cellina$_{node-pert}$      0.657693  0.210194  0.795482  0.168933   
SpatialProp$_{node-pert}$  0.599155  0.153979  0.704735  0.134759   

                          direction_match_k                rmse            \
                                       mean       std      mean       std   
model_name                                                                  
mean shift                         0.159333  0.075284  3.447524  0.333883   
CPA                                0.422000  0.131069  3.615080  0.353036   
scGen                              0.208667  0.126135  3.546663  0.363213   
SpatialProp                        0.082667  0.074799  3.460886  0.392315   
MintFlow                           0.286000  0.161610  3.601694  0.369724   
cellina (ablated)                  0.390667  0.140551  3.631372  0.351855   
cellina-GAT                        0.525714  0.140112  3.597029  0.359406   
cellina                            0.520667  0.175612  3.614027  0.352948   
cellina$_{node-pert}$              0.316000  0.174421  3.630757  0.353278   
SpatialProp$_{node-pert}$          0.082667  0.074799  3.460048  0.391607   

                            rmse_lfc            
                                mean       std  
model_name                                      
mean shift                 12.571237  2.550779  
CPA                         6.401187  4.755085  
scGen                       6.525090  4.054144  
SpatialProp                 6.691006  2.776821  
MintFlow                    7.238124  5.409321  
cellina (ablated)           6.690000  4.620697  
cellina-GAT                 5.759385  4.485479  
cellina                     6.052597  5.040709  
cellina$_{node-pert}$       6.880412  4.611413  
SpatialProp$_{node-pert}$   6.692856  2.779137

In [17]:
def find_best(agg: pd.DataFrame) -> dict:
    best = {}
    if isinstance(agg.index, pd.MultiIndex):
        # agg index: (perturbation, model_name)
        # collect perturbations preserving order
        raw = [p for p, _ in agg.index.tolist()]
        seen = set()
        perturbations = [p for p in raw if not (p in seen or seen.add(p))]
        for metric, direction in METRICS.items():
            for pert in perturbations:
                try:
                    means = agg.loc[pert][(metric, "mean")].dropna()
                except KeyError:
                    continue
                if means.empty:
                    continue
                best[(pert, metric)] = means.idxmax() if direction > 0 else means.idxmin()
    else:
        for metric, direction in METRICS.items():
            means = agg[(metric, "mean")].dropna()
            if means.empty:
                continue
            best[metric] = means.idxmax() if direction > 0 else means.idxmin()
    return best


def format_cell(mean, std, bold, na_str="--"):
    if pd.isna(mean):
        return na_str
    s = f"{mean:.2f} $\\pm$ {std:.2f}" if not pd.isna(std) else f"{mean:.2f}"
    return f"\\textbf{{{s}}}" if bold else s

In [18]:
best = find_best(df_avg)
table = pd.DataFrame(index=df_avg.index, columns=[PRETTY_METRIC[m] for m in METRICS])

for metric in METRICS:
    col = PRETTY_METRIC[metric]
    for model in df_avg.index:
        model_name = model

        mean = df_avg.loc[model, (metric, "mean")]
        std = df_avg.loc[model, (metric, "std")]

        bold = (best.get(metric) == model_name)

        table.loc[model, col] = format_cell(mean, std, bold)
table

,Spearman $\uparrow$,Pearson $\uparrow$,Signed Precision $\uparrow$,RMSE\textsubscript{counts} $\downarrow$,RMSE\textsubscript{LFC} $\downarrow$
model_name,,,,,
mean shift,0.47 $\pm$ 0.23,0.47 $\pm$ 0.22,0.16 $\pm$ 0.08,\textbf{3.45 $\pm$ 0.33},12.57 $\pm$ 2.55
CPA,0.68 $\pm$ 0.19,0.82 $\pm$ 0.16,0.42 $\pm$ 0.13,3.62 $\pm$ 0.35,6.40 $\pm$ 4.76
scGen,0.61 $\pm$ 0.21,0.77 $\pm$ 0.17,0.21 $\pm$ 0.13,3.55 $\pm$ 0.36,6.53 $\pm$ 4.05
SpatialProp,0.60 $\pm$ 0.16,0.71 $\pm$ 0.14,0.08 $\pm$ 0.07,3.46 $\pm$ 0.39,6.69 $\pm$ 2.78
MintFlow,0.66 $\pm$ 0.21,0.81 $\pm$ 0.17,0.29 $\pm$ 0.16,3.60 $\pm$ 0.37,7.24 $\pm$ 5.41
cellina (ablated),0.69 $\pm$ 0.19,0.82 $\pm$ 0.15,0.39 $\pm$ 0.14,3.63 $\pm$ 0.35,6.69 $\pm$ 4.62
cellina-GAT,\textbf{0.75 $\pm$ 0.13},\textbf{0.85 $\pm$ 0.15},\textbf{0.53 $\pm$ 0.14},3.60 $\pm$ 0.36,\textbf{5.76 $\pm$ 4.49}
cellina,0.73 $\pm$ 0.20,0.84 $\pm$ 0.16,0.52 $\pm$ 0.18,3.61 $\pm$ 0.35,6.05 $\pm$ 5.04
cellina$_{node-pert}$,0.66 $\pm$ 0.21,0.80 $\pm$ 0.17,0.32 $\pm$ 0.17,3.63 $\pm$ 0.35,6.88 $\pm$ 4.61


In [19]:
flat = table.reset_index(level=0, drop=False)

latex = flat.to_latex(
    index=False,
    index_names=True,
    escape=False,
    column_format="l" + "c" * flat.shape[1],
    caption=(
        f"Leave-one-celltype-out performance (top {N_DEG} DEGs). "
        "Mean $\\pm$ std across slides. Best per metric in \\textbf{bold}."
    ),
    label=f"tab:loo_summary_{DATASET_NAME}",
)
latex = latex.replace("\\begin{table}", "\\begin{table}[t]\n\\centering")

print(latex)

(OUT_DIR / f"loo_summary_{DATASET_NAME}_DEG_{N_DEG}_table_avg.tex").write_text(latex)

\begin{table}[t]
\centering
\caption{Leave-one-celltype-out performance (top 50 DEGs). Mean $\pm$ std across slides. Best per metric in \textbf{bold}.}
\label{tab:loo_summary_merfish}
\begin{tabular}{lcccccc}
\toprule
model_name & Spearman $\uparrow$ & Pearson $\uparrow$ & Signed Precision $\uparrow$ & RMSE\textsubscript{counts} $\downarrow$ & RMSE\textsubscript{LFC} $\downarrow$ \\
\midrule
mean shift & 0.47 $\pm$ 0.23 & 0.47 $\pm$ 0.22 & 0.16 $\pm$ 0.08 & \textbf{3.45 $\pm$ 0.33} & 12.57 $\pm$ 2.55 \\
CPA & 0.68 $\pm$ 0.19 & 0.82 $\pm$ 0.16 & 0.42 $\pm$ 0.13 & 3.62 $\pm$ 0.35 & 6.40 $\pm$ 4.76 \\
scGen & 0.61 $\pm$ 0.21 & 0.77 $\pm$ 0.17 & 0.21 $\pm$ 0.13 & 3.55 $\pm$ 0.36 & 6.53 $\pm$ 4.05 \\
SpatialProp & 0.60 $\pm$ 0.16 & 0.71 $\pm$ 0.14 & 0.08 $\pm$ 0.07 & 3.46 $\pm$ 0.39 & 6.69 $\pm$ 2.78 \\
MintFlow & 0.66 $\pm$ 0.21 & 0.81 $\pm$ 0.17 & 0.29 $\pm$ 0.16 & 3.60 $\pm$ 0.37 & 7.24 $\pm$ 5.41 \\
cellina (ablated) & 0.69 $\pm$ 0.19 & 0.82 $\pm$ 0.15 & 0.39 $\pm$ 0.14 & 3.63 $\pm$ 0.3

1537